# 4.3 Relationships in Data — Enhanced Tutorial

**Primary outcome:** Apply advanced relationship-modelling techniques — interaction terms, partial correlation, polynomial/non-linear regression, logistic regression as a relationship model, and time-series correlation — using real datasets.

**Estimated time:** 90–120 minutes

**Prerequisites:** Complete `tutorial.ipynb` (base tutorial) before working through this notebook.

---

## 1. Quick Recap

The table below summarises what the base tutorial covered versus what this notebook adds.

| Topic | Base Tutorial | This Notebook |
|---|---|---|
| Types of relationships | Linear, non-linear, none, negative | already covered |
| Pearson correlation | Full coverage | already covered |
| Spearman correlation | Full coverage | already covered |
| Simple linear regression | Full coverage incl. residuals | already covered |
| Multiple linear regression | Full coverage | already covered |
| VIF / multicollinearity | Covered | already covered |
| **Partial correlation** | Mentioned | **Implemented manually here** |
| **Interaction terms** | Not covered | **New in this notebook** |
| **Polynomial regression** | Mentioned briefly | **Full pipeline + degree comparison** |
| **Log transformation** | Not covered | **New in this notebook** |
| **Robust regression** | Not covered | **OLS vs Huber vs RANSAC** |
| **Logistic regression as relationship model** | Not covered | **Odds ratios + S-curve** |
| **Autocorrelation / time-series** | Not covered | **ACF + seasonality** |
| **Cross-correlation** | Not covered | **Leading/lagging indicators** |
| **Regularised regression (Ridge / Lasso)** | Not covered | **Coefficient paths + feature selection** |
| **Model comparison dashboard** | Not covered | **Summary table + predicted vs actual** |


In [ ]:
# All imports for this notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# sklearn
from sklearn.linear_model import (
    LinearRegression, LogisticRegression,
    Ridge, Lasso, HuberRegressor, RANSACRegressor
)
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.datasets import load_diabetes

# statsmodels
from statsmodels.graphics.tsaplots import plot_acf

# reproducibility & aesthetics
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

print('Libraries loaded successfully.')


In [ ]:
# Load datasets
tips    = sns.load_dataset('tips')
mpg     = sns.load_dataset('mpg').dropna(subset=['horsepower'])
flights = sns.load_dataset('flights')

# diabetes (sklearn)
diab_data = load_diabetes()
diabetes  = pd.DataFrame(diab_data.data, columns=diab_data.feature_names)
diabetes['target'] = diab_data.target

print('tips    :', tips.shape)
print('mpg     :', mpg.shape)
print('flights :', flights.shape)
print('diabetes:', diabetes.shape)


---

## 2. Partial Correlation

### Motivation

You have calculated the raw Pearson correlation between `size` (party size) and `tip`. But larger parties also tend to have larger bills — and larger bills clearly produce larger tips. **Partial correlation** asks:

> *After controlling for total bill, does party size still predict tip independently?*

### Method — manual implementation

1. Regress `total_bill` out of `tip` — get residuals `e_tip`
2. Regress `total_bill` out of `size` — get residuals `e_size`
3. Correlate `e_tip` and `e_size` — this is the partial correlation

This removes the shared variance explained by `total_bill`, leaving only the unique relationship between `size` and `tip`.


In [ ]:
# Step 1: raw correlations
r_raw_tip_bill  = tips['tip'].corr(tips['total_bill'])
r_raw_size_bill = tips['size'].corr(tips['total_bill'])
r_raw_size_tip  = tips['size'].corr(tips['tip'])

print('Raw correlations:')
print(f'  tip   vs total_bill : {r_raw_tip_bill:.3f}')
print(f'  size  vs total_bill : {r_raw_size_bill:.3f}')
print(f'  size  vs tip        : {r_raw_size_tip:.3f}  <- this is what we want to purify')


In [ ]:
# Step 2: manual partial correlation (regress out total_bill)
X_bill = tips[['total_bill']].values

# Residuals of tip ~ total_bill
lr1   = LinearRegression().fit(X_bill, tips['tip'])
e_tip = tips['tip'].values - lr1.predict(X_bill)

# Residuals of size ~ total_bill
lr2    = LinearRegression().fit(X_bill, tips['size'])
e_size = tips['size'].values - lr2.predict(X_bill)

# Partial correlation = correlation of the residuals
r_partial, p_partial = stats.pearsonr(e_size, e_tip)

print('Partial correlation (size -> tip, controlling for total_bill):')
print(f'  r = {r_partial:.3f},  p = {p_partial:.4f}')
print()
print('Comparison:')
print(f'  Raw correlation   size vs tip : {r_raw_size_tip:.3f}')
print(f'  Partial corr.     size vs tip : {r_partial:.3f}')


In [ ]:
# Visualise: raw scatter vs residual scatter
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw
axes[0].scatter(tips['size'], tips['tip'], alpha=0.5, edgecolors='k', linewidths=0.3)
m, b = np.polyfit(tips['size'], tips['tip'], 1)
xs = np.linspace(tips['size'].min(), tips['size'].max(), 100)
axes[0].plot(xs, m*xs + b, 'r-', linewidth=2)
axes[0].set_xlabel('Party size')
axes[0].set_ylabel('Tip ($)')
axes[0].set_title(f'Raw correlation\nr = {r_raw_size_tip:.3f}')

# Partial (residuals)
axes[1].scatter(e_size, e_tip, alpha=0.5, edgecolors='k', linewidths=0.3)
m2, b2 = np.polyfit(e_size, e_tip, 1)
xs2 = np.linspace(e_size.min(), e_size.max(), 100)
axes[1].plot(xs2, m2*xs2 + b2, 'r-', linewidth=2)
axes[1].set_xlabel('size residuals (bill removed)')
axes[1].set_ylabel('tip residuals (bill removed)')
axes[1].set_title(f'Partial correlation\nr = {r_partial:.3f}')

plt.suptitle('Partial Correlation: size -> tip, controlling for total_bill', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### Interpretation

The **raw correlation** between party size and tip is moderate and positive (~0.49). However, this is partly because bigger parties order more food (larger bills), and larger bills attract larger tips.

After controlling for `total_bill`, the **partial correlation drops noticeably**. This tells us that much of the apparent relationship between party size and tip was actually driven by bill size — the confounding variable. The remaining partial correlation represents the unique, direct effect of having more people at the table on the tip amount, independent of the bill.

> **Rule of thumb:** Always ask whether a correlation might be inflated by a third variable. Partial correlation is your tool for checking.


---

## 3. Interaction Terms

### Hypothesis

> *Does the relationship between bill size and tip **differ** by time of day (lunch vs dinner)?*

A standard MLR model assumes that the slope of `total_bill -> tip` is the same for both lunch and dinner. An **interaction term** relaxes this assumption, allowing the slope to differ across groups.

**Model without interaction:**

tip = b0 + b1 * total_bill + b2 * time

**Model with interaction:**

tip = b0 + b1 * total_bill + b2 * time + b3 * (total_bill x time)

If b3 is significant, the **effect of bill size on tip is different at lunch vs dinner**.


In [ ]:
# Prepare features
tips_int = tips.copy()
tips_int['time_enc']   = (tips_int['time'] == 'Dinner').astype(int)  # 0=Lunch, 1=Dinner
tips_int['interaction'] = tips_int['total_bill'] * tips_int['time_enc']

y_tips = tips_int['tip'].values

# Model A: no interaction
X_no  = tips_int[['total_bill', 'time_enc']].values
lr_no = LinearRegression().fit(X_no, y_tips)
r2_no = r2_score(y_tips, lr_no.predict(X_no))

# Model B: with interaction
X_int  = tips_int[['total_bill', 'time_enc', 'interaction']].values
lr_int = LinearRegression().fit(X_int, y_tips)
r2_int = r2_score(y_tips, lr_int.predict(X_int))

print('Model without interaction:')
print(f'  intercept={lr_no.intercept_:.3f}, bill={lr_no.coef_[0]:.3f}, time={lr_no.coef_[1]:.3f}')
print(f'  R2 = {r2_no:.4f}')
print()
print('Model WITH interaction:')
print(f'  intercept={lr_int.intercept_:.3f}, bill={lr_int.coef_[0]:.3f}, '
      f'time={lr_int.coef_[1]:.3f}, bill x time={lr_int.coef_[2]:.3f}')
print(f'  R2 = {r2_int:.4f}')
print()
print(f'  R2 improvement from interaction: {r2_int - r2_no:.4f}')


In [ ]:
# Plot: separate regression lines per time of day
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = {'Lunch': '#e07b39', 'Dinner': '#3a7ebf'}

for ax, (model, label, r2) in zip(axes,
    [(lr_no,  'Without Interaction', r2_no),
     (lr_int, 'With Interaction',    r2_int)]):

    for time_val, time_enc, color in [('Lunch', 0, colors['Lunch']), ('Dinner', 1, colors['Dinner'])]:
        mask = tips_int['time'] == time_val
        ax.scatter(tips_int.loc[mask, 'total_bill'], tips_int.loc[mask, 'tip'],
                   alpha=0.5, color=color, label=time_val, edgecolors='k', linewidths=0.3, s=40)
        x_line = np.linspace(tips_int['total_bill'].min(), tips_int['total_bill'].max(), 100)
        if label == 'Without Interaction':
            X_line = np.column_stack([x_line, np.full(100, time_enc)])
        else:
            X_line = np.column_stack([x_line, np.full(100, time_enc), x_line * time_enc])
        ax.plot(x_line, model.predict(X_line), color=color, linewidth=2.5)

    ax.set_xlabel('Total Bill ($)')
    ax.set_ylabel('Tip ($)')
    ax.set_title(f'{label}\nR2 = {r2:.4f}')
    ax.legend(title='Time')

plt.suptitle('Interaction Terms: Does bill->tip slope differ by time of day?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### Interpretation

In the **without-interaction** model, the two regression lines are **parallel** — the model assumes lunch and dinner diners tip at the same rate per dollar of bill.

In the **with-interaction** model, the lines can have **different slopes**. A non-zero interaction coefficient means the relationship between bill size and tip is not the same across meal times.

**When does an interaction matter?**
- When you suspect the effect of X on Y depends on the level of a third variable Z.
- Common in social science ("the effect of education on income differs by gender"), medicine ("treatment effect varies by age group"), and marketing ("ad effectiveness differs by channel").
- Always check: does the R2 improvement justify the added model complexity?


---

## 4. Polynomial Regression

Linear regression assumes a straight-line relationship. But what if the relationship is curved? **Polynomial regression** fits a curve by adding powers of the predictor as extra features:

y_hat = b0 + b1*x + b2*x^2 + b3*x^3 + ...

We'll use the `mpg` dataset to explore whether `horsepower -> mpg` is better described by a curve than a line.


In [ ]:
# Inspect the relationship: horsepower vs mpg
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(mpg['horsepower'], mpg['mpg'], alpha=0.5, edgecolors='k', linewidths=0.3, s=30)
axes[0].set_xlabel('Horsepower')
axes[0].set_ylabel('MPG')
axes[0].set_title('Raw scatter: horsepower vs mpg')

# Linear fit residuals
X_hp = mpg[['horsepower']].values
y_hp = mpg['mpg'].values
lr_lin = LinearRegression().fit(X_hp, y_hp)
resid_lin = y_hp - lr_lin.predict(X_hp)

axes[1].scatter(lr_lin.predict(X_hp), resid_lin, alpha=0.5, edgecolors='k', linewidths=0.3, s=30)
axes[1].axhline(0, color='red', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Fitted values (linear model)')
axes[1].set_ylabel('Residuals')
axes[1].set_title('Residuals of linear fit\n(pattern = non-linearity present)')

plt.suptitle('Is horsepower -> mpg truly linear?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Linear fit R2: {r2_score(y_hp, lr_lin.predict(X_hp)):.3f}')


In [ ]:
# Fit polynomial degrees 1, 2, 3
X_train_hp, X_test_hp, y_train_hp, y_test_hp = train_test_split(
    X_hp, y_hp, test_size=0.2, random_state=42)

poly_results = {}
for deg in [1, 2, 3]:
    pipe = Pipeline([
        ('poly', PolynomialFeatures(degree=deg, include_bias=False)),
        ('lr',   LinearRegression())
    ])
    pipe.fit(X_train_hp, y_train_hp)
    r2_tr = r2_score(y_train_hp, pipe.predict(X_train_hp))
    r2_te = r2_score(y_test_hp,  pipe.predict(X_test_hp))
    poly_results[deg] = {'model': pipe, 'r2_train': r2_tr, 'r2_test': r2_te}
    print(f'Degree {deg}: train R2={r2_tr:.4f}  |  test R2={r2_te:.4f}')

best_deg = max(poly_results, key=lambda d: poly_results[d]['r2_test'])
print(f'\nBest degree by test R2: {best_deg}')


In [ ]:
# Plot all three fits overlaid
fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(X_hp, y_hp, alpha=0.4, color='#aaaaaa',
           edgecolors='k', linewidths=0.2, s=30, label='Data')

x_curve = np.linspace(X_hp.min(), X_hp.max(), 300).reshape(-1, 1)
colors_deg = {1: '#e07b39', 2: '#3a7ebf', 3: '#2ca02c'}

for deg in [1, 2, 3]:
    y_curve = poly_results[deg]['model'].predict(x_curve)
    r2t = poly_results[deg]['r2_test']
    lw = 3 if deg == best_deg else 1.5
    ls = '-' if deg == best_deg else '--'
    label = f'Degree {deg} (test R2={r2t:.3f})' + (' <- best' if deg == best_deg else '')
    ax.plot(x_curve, y_curve, color=colors_deg[deg], linewidth=lw, linestyle=ls, label=label)

ax.set_xlabel('Horsepower')
ax.set_ylabel('MPG')
ax.set_title('Polynomial Regression: Degree 1, 2, 3 on mpg dataset')
ax.legend()
plt.tight_layout()
plt.show()


### Interpretation

The **residual plot** for the linear fit shows a clear curved pattern — positive residuals at low and high horsepower, negative residuals in the middle. This is the signature of non-linearity that a linear model cannot capture.

Adding a **quadratic term (degree 2)** typically captures most of the curvature in the `horsepower -> mpg` relationship and produces the best test R2.

A **degree-3 fit** may not improve test R2 much — and with real data, it risks **overfitting**: it hugs the training data more tightly but generalises less well.

> **Rule of thumb:** Start with degree 2. Only go higher if (a) residual plots still show systematic curves, and (b) test R2 genuinely improves. Watch the gap between train R2 and test R2 — a widening gap signals overfitting.


---

## 5. Non-Linear Regression: Log Transformation

Sometimes a simple mathematical transformation of the predictor (or response) straightens out a curved relationship. The **log transformation** is particularly useful when:

- The predictor is right-skewed (many small values, a few very large ones)
- The relationship looks like exponential decay or logarithmic growth
- We want to model *proportional* rather than *absolute* changes

We'll compare `weight -> mpg` (raw) versus `log(weight) -> mpg` (transformed).


In [ ]:
# Fit linear model: weight -> mpg vs log(weight) -> mpg
mpg_w = mpg.dropna(subset=['weight', 'mpg']).copy()
mpg_w['log_weight'] = np.log(mpg_w['weight'])

X_w  = mpg_w[['weight']].values
X_lw = mpg_w[['log_weight']].values
y_w  = mpg_w['mpg'].values

lr_raw = LinearRegression().fit(X_w,  y_w)
lr_log = LinearRegression().fit(X_lw, y_w)

r2_raw_w = r2_score(y_w, lr_raw.predict(X_w))
r2_log_w = r2_score(y_w, lr_log.predict(X_lw))

print(f'Linear model    (weight -> mpg)      : R2 = {r2_raw_w:.4f}')
print(f'Log-transformed (log(weight) -> mpg) : R2 = {r2_log_w:.4f}')
print(f'Improvement from log transform: {r2_log_w - r2_raw_w:+.4f}')


In [ ]:
# Plot both models
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

w_line = np.linspace(mpg_w['weight'].min(), mpg_w['weight'].max(), 300)

# Left: original scale, both fits
axes[0].scatter(mpg_w['weight'], mpg_w['mpg'],
                alpha=0.4, edgecolors='k', linewidths=0.2, s=30)
axes[0].plot(w_line, lr_raw.predict(w_line.reshape(-1,1)),
             'r-', linewidth=2, label=f'Linear  R2={r2_raw_w:.3f}')
axes[0].plot(w_line, lr_log.predict(np.log(w_line).reshape(-1,1)),
             'b-', linewidth=2, label=f'Log-transform R2={r2_log_w:.3f}')
axes[0].set_xlabel('Vehicle Weight (lbs)')
axes[0].set_ylabel('MPG')
axes[0].set_title('weight vs mpg')
axes[0].legend()

# Right: log scale x-axis (linearised)
axes[1].scatter(mpg_w['log_weight'], mpg_w['mpg'],
                alpha=0.4, edgecolors='k', linewidths=0.2, s=30)
lw_line = np.linspace(mpg_w['log_weight'].min(), mpg_w['log_weight'].max(), 300)
axes[1].plot(lw_line, lr_log.predict(lw_line.reshape(-1,1)), 'b-', linewidth=2)
axes[1].set_xlabel('log(Weight)')
axes[1].set_ylabel('MPG')
axes[1].set_title(f'log(weight) vs mpg — linearised\nR2 = {r2_log_w:.4f}')

plt.suptitle('Log Transformation: Straightening a Curved Relationship',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### Interpretation

The right-hand plot shows that `log(weight) -> mpg` is much closer to a straight line. The log transformation **compresses the right tail** of the weight distribution, stretching out the lower end where most of the interesting variation happens.

The R2 improvement confirms the log-transformed model fits the data better.

**How to interpret the log-transformed coefficient:**
A one-unit increase in `log(weight)` corresponds to a proportional increase in weight. If the coefficient is b, then a 1% increase in weight is associated with approximately b/100 units change in mpg.

> **When to use log transformation:** Try it when your predictor is skewed, when residual plots show a fan shape (heteroscedasticity), or when theory suggests a multiplicative rather than additive effect.


---

## 6. Robust Regression: Handling Outliers

Standard OLS minimises the **sum of squared residuals** — which means a single extreme outlier can drag the regression line substantially. **Robust regression** methods are resistant to outliers:

| Method | How it works | Best for |
|---|---|---|
| OLS | Minimise squared errors (outliers have high influence) | Clean data |
| HuberRegressor | Squared loss for small residuals, linear loss for large ones | Moderate outliers |
| RANSACRegressor | Randomly sample inliers, fit on best consensus set | Heavy contamination |

We will **inject 10 artificial outliers** into the tips dataset and compare all three.


In [ ]:
# Inject outliers
np.random.seed(42)
tips_dirty = tips.copy()

# Set 10 small-bill rows to have a massive tip
outlier_idx = tips_dirty[tips_dirty['total_bill'] < 8].sample(10, random_state=42).index
tips_dirty.loc[outlier_idx, 'tip'] = 20.0

X_d = tips_dirty[['total_bill']].values
y_d = tips_dirty['tip'].values

# Fit three models
ols    = LinearRegression().fit(X_d, y_d)
huber  = HuberRegressor(epsilon=1.35).fit(X_d, y_d)
ransac = RANSACRegressor(random_state=42).fit(X_d, y_d)

print('Model coefficients (intercept, slope):')
print(f'  OLS    : intercept={ols.intercept_:.3f},   slope={ols.coef_[0]:.3f}')
print(f'  Huber  : intercept={huber.intercept_:.3f},   slope={huber.coef_[0]:.3f}')
print(f'  RANSAC : intercept={ransac.estimator_.intercept_:.3f},   slope={ransac.estimator_.coef_[0]:.3f}')


In [ ]:
# Plot all three regression lines
fig, ax = plt.subplots(figsize=(9, 5))

is_outlier = tips_dirty.index.isin(outlier_idx)
ax.scatter(X_d[~is_outlier], y_d[~is_outlier], alpha=0.5, color='#aaaaaa',
           edgecolors='k', linewidths=0.2, s=35, label='Normal points')
ax.scatter(X_d[is_outlier], y_d[is_outlier], alpha=0.9, color='#d62728',
           edgecolors='k', linewidths=0.5, s=80, marker='*',
           label='Injected outliers', zorder=5)

x_line = np.linspace(X_d.min(), X_d.max(), 200).reshape(-1, 1)
ax.plot(x_line, ols.predict(x_line),    'r-',  linewidth=2,   label='OLS (pulled by outliers)')
ax.plot(x_line, huber.predict(x_line),  'b--', linewidth=2,   label='Huber (robust)')
ax.plot(x_line, ransac.predict(x_line), 'g:',  linewidth=2.5, label='RANSAC (robust)')

ax.set_xlabel('Total Bill ($)')
ax.set_ylabel('Tip ($)')
ax.set_title('Robust Regression: OLS vs Huber vs RANSAC\n(10 artificial outliers injected)')
ax.legend()
plt.tight_layout()
plt.show()


### Interpretation

The injected outliers (small bills, $20 tips) are clearly anomalous. Look at how each model responds:

- **OLS** is noticeably pulled upward — its intercept is inflated and its slope is distorted because squared-error loss penalises large residuals disproportionately.
- **HuberRegressor** uses a linear penalty for large residuals, limiting the influence of extreme points. The line is closer to the true relationship.
- **RANSACRegressor** identifies the largest consensus set of inlier points and ignores the rest entirely. It recovers the true underlying relationship most faithfully when outliers are numerous.

> **When to use robust regression:**
> - Use **Huber** when you expect a moderate proportion of outliers due to measurement error.
> - Use **RANSAC** when you suspect a distinct subgroup of data points follows a completely different pattern.
> - Always visualise your residuals before and after — robust regression is not a substitute for understanding *why* outliers exist.


---

## 7. Logistic Regression as a Relationship Model

So far we have used regression to model a **continuous outcome** (tip amount, mpg). But regression is a general framework for modelling relationships. **Logistic regression** models the probability of a binary outcome.

**Question:** *Does total bill predict whether a customer is a generous tipper (tip rate > 20%)?*

The logistic model estimates:

P(generous = 1 | total_bill) = 1 / (1 + exp(-(b0 + b1 * total_bill)))

The coefficient b1 is interpreted via the **odds ratio**: exp(b1).


In [ ]:
# Create binary target: generous tipper = tip rate > 20%
tips_lr = tips.copy()
tips_lr['tip_rate'] = tips_lr['tip'] / tips_lr['total_bill']
tips_lr['generous'] = (tips_lr['tip_rate'] > 0.20).astype(int)

print(f'Generous tippers (tip rate > 20%): {tips_lr["generous"].sum()} out of {len(tips_lr)}')
print(f'Base rate: {tips_lr["generous"].mean():.1%}')


In [ ]:
# Fit logistic regression
X_lr = tips_lr[['total_bill']].values
y_lr = tips_lr['generous'].values

log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_lr, y_lr)

beta_0     = log_reg.intercept_[0]
beta_1     = log_reg.coef_[0][0]
odds_ratio = np.exp(beta_1)
auc        = roc_auc_score(y_lr, log_reg.predict_proba(X_lr)[:, 1])

print(f'Coefficients:  b0 = {beta_0:.4f},  b1 = {beta_1:.4f}')
print(f'Odds ratio (exp(b1)):  {odds_ratio:.4f}')
print(f'AUC-ROC: {auc:.4f}')
print()
if odds_ratio < 1:
    pct = (1 - odds_ratio) * 100
    print(f'Interpretation: A $1 increase in total_bill is associated with a {pct:.1f}% DECREASE')
    print('in the odds of being a generous tipper.')
else:
    pct = (odds_ratio - 1) * 100
    print(f'Interpretation: A $1 increase in total_bill is associated with a {pct:.1f}% INCREASE')
    print('in the odds of being a generous tipper.')


In [ ]:
# Plot S-curve: probability of being generous vs total_bill
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x_range  = np.linspace(tips_lr['total_bill'].min() - 2,
                        tips_lr['total_bill'].max() + 2, 300).reshape(-1, 1)
prob_curve = log_reg.predict_proba(x_range)[:, 1]

# S-curve
np.random.seed(42)
for label, val, color in [('Not generous', 0, '#e07b39'), ('Generous', 1, '#3a7ebf')]:
    mask = y_lr == val
    axes[0].scatter(
        X_lr[mask],
        y_lr[mask] + np.random.normal(0, 0.02, mask.sum()),
        alpha=0.3, color=color, s=25, label=label
    )

axes[0].plot(x_range, prob_curve, 'k-', linewidth=2.5, label='P(generous | bill)')
axes[0].axhline(0.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
axes[0].set_xlabel('Total Bill ($)')
axes[0].set_ylabel('P(generous tipper)')
axes[0].set_title('Logistic S-curve\nP(tip > 20%) vs Total Bill')
axes[0].legend(fontsize=8)
axes[0].set_ylim(-0.1, 1.1)

# Tip rate scatter
sc = axes[1].scatter(tips_lr['total_bill'], tips_lr['tip_rate'],
                     c=tips_lr['generous'], cmap='coolwarm',
                     alpha=0.6, edgecolors='k', linewidths=0.2, s=35)
axes[1].axhline(0.20, color='black', linestyle='--', linewidth=1.5, label='20% threshold')
axes[1].set_xlabel('Total Bill ($)')
axes[1].set_ylabel('Tip Rate (tip / bill)')
axes[1].set_title('Tip Rate vs Total Bill\n(red = generous, blue = not generous)')
axes[1].legend()

plt.suptitle('Logistic Regression as a Relationship Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### Interpretation

The S-curve shows how the probability of being a generous tipper changes as the bill grows. The **negative coefficient** suggests that higher bills are associated with lower tip rates — people with large bills tend to tip proportionally less.

**Odds ratio < 1** means each additional dollar on the bill slightly *decreases* the odds of tipping above 20%.

**Regression as a general framework:**
- OLS regression: How does X affect the *mean* of a continuous Y?
- Logistic regression: How does X affect the *probability* of a binary Y?
- Both model the **relationship** between variables — they differ only in the type of outcome and the link function.
- The coefficient interpretation (odds ratio vs slope) changes, but the core logic — fitting a systematic relationship and measuring how well it explains the data — is identical.


---

## 8. Time-Series Correlation: Autocorrelation

So far, we have looked at relationships *between* different variables. But a variable can also be correlated with **itself at different points in time** — this is called **autocorrelation**.

**Autocorrelation at lag k:** How correlated is y_t with y_{t-k}?

Strong autocorrelation implies that past values are predictive of future values — which has major implications for forecasting and model assumption checks.

We will use the `flights` dataset: monthly airline passengers from 1949 to 1960.


In [ ]:
# Prepare flights time series
flights_ts = flights.pivot(index='year', columns='month', values='passengers')\
                    .stack().reset_index()
flights_ts.columns = ['year', 'month', 'passengers']

month_map = {'Jan':1,'Feb':2,'Mar':3,'Apr':4,'May':5,'Jun':6,
             'Jul':7,'Aug':8,'Sep':9,'Oct':10,'Nov':11,'Dec':12}
flights_ts['month_num'] = flights_ts['month'].map(month_map)
flights_ts = flights_ts.sort_values(['year', 'month_num']).reset_index(drop=True)

passengers = pd.Series(
    flights_ts['passengers'].values,
    index=pd.date_range('1949-01', periods=len(flights_ts), freq='ME')
)

ac_lag1  = passengers.autocorr(lag=1)
ac_lag12 = passengers.autocorr(lag=12)

print(f'Autocorrelation at lag  1 (1 month apart):  {ac_lag1:.3f}')
print(f'Autocorrelation at lag 12 (1 year apart):   {ac_lag12:.3f}')
print()
print('Interpretation:')
print(f'  Lag 1  r={ac_lag1:.3f}: Passengers this month are strongly correlated with last month.')
print(f'  Lag 12 r={ac_lag12:.3f}: Passengers this month are strongly correlated with same month last year.')
print('  -> Strong seasonal pattern (annual cycle).')


In [ ]:
# Plot time series + ACF
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Time series
axes[0].plot(passengers.index, passengers.values, color='#3a7ebf', linewidth=1.5)
axes[0].fill_between(passengers.index, passengers.values, alpha=0.15, color='#3a7ebf')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Passengers (thousands)')
axes[0].set_title('Monthly Airline Passengers 1949-1960')

# ACF
plot_acf(passengers.values, lags=36, ax=axes[1], color='#3a7ebf', alpha=0.05)
axes[1].set_title('Autocorrelation Function (ACF) — lags 0 to 36 months')
axes[1].set_xlabel('Lag (months)')
axes[1].set_ylabel('Autocorrelation')
axes[1].axvline(12, color='red',    linestyle='--', linewidth=1, alpha=0.7, label='Lag 12 (annual)')
axes[1].axvline(24, color='orange', linestyle='--', linewidth=1, alpha=0.7, label='Lag 24 (2 years)')
axes[1].legend(fontsize=9)

plt.suptitle('Time-Series Autocorrelation: Flights Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### Interpretation

The **ACF plot** shows bars for each lag. The blue shaded region is the **95% confidence band** — bars extending beyond it are statistically significant.

**What we see in the flights data:**
- Bars remain **very high and significant** for many lags — this reflects a strong upward trend (non-stationarity).
- There are **peaks at lags 12, 24, 36** — strong annual seasonality. Passenger numbers in January 1955 closely resemble January 1954 and January 1956.
- The gradual decline in the ACF is the signature of a **trend plus seasonality** combined.

**Why does this matter for regression?**
Standard regression assumes residuals are **independent** (no autocorrelation). If you fit a regression on time-series data and your residuals are autocorrelated, your standard errors are wrong, p-values are unreliable, and your model is missing structure. Strong autocorrelation at lag 12 is the mathematical fingerprint of **seasonality** — and a cue to use seasonal models (SARIMA, seasonal decomposition, etc.).


---

## 9. Cross-Correlation Between Two Series

**Autocorrelation** measures a series' relationship with itself at past lags. **Cross-correlation** measures the correlation between *two different series* at various lags. This is powerful for detecting **leading and lagging indicators**:

> *If marketing spend this month is correlated with sales 2 months from now, then marketing spend 'leads' sales by 2 months.*

We will construct a synthetic example where this relationship is built in, then detect it using cross-correlation.


In [ ]:
# Construct synthetic leading/lagging series
np.random.seed(42)
n_months = 60
t = np.arange(n_months)

# Marketing spend: seasonal with upward trend + noise
marketing = 100 + 2*t + 20*np.sin(2*np.pi*t/12) + np.random.normal(0, 8, n_months)

# Sales: follows marketing with a 2-month lag
lag_true = 2
sales = np.zeros(n_months)
sales[:lag_true] = 200
sales[lag_true:] = 1.5 * marketing[:-lag_true] + 50 + np.random.normal(0, 15, n_months - lag_true)

dates = pd.date_range('2020-01', periods=n_months, freq='ME')

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# Plot both series
axes[0].plot(dates, marketing, color='#e07b39', linewidth=1.5, label='Marketing Spend')
ax_twin = axes[0].twinx()
ax_twin.plot(dates, sales, color='#3a7ebf', linewidth=1.5, label='Sales')
axes[0].set_ylabel('Spend ($)')
ax_twin.set_ylabel('Sales ($)')
axes[0].set_title(f'Marketing Spend vs Sales (sales lags spend by {lag_true} months)')
lines1, labels1 = axes[0].get_legend_handles_labels()
lines2, labels2 = ax_twin.get_legend_handles_labels()
axes[0].legend(lines1+lines2, labels1+labels2, loc='upper left')

# Cross-correlation
m_norm = (marketing - marketing.mean()) / (marketing.std() * n_months)
s_norm = (sales     - sales.mean())     /  sales.std()
xcorr  = np.correlate(s_norm, m_norm, mode='full')
lags   = np.arange(-(n_months-1), n_months)

axes[1].plot(lags, xcorr, color='#9467bd', linewidth=1.5)
peak_lag = lags[np.argmax(xcorr)]
axes[1].axvline(peak_lag, color='red', linestyle='--', linewidth=2,
                label=f'Peak at lag {peak_lag} -> marketing leads sales by {peak_lag} months')
axes[1].axvline(0, color='gray', linestyle=':', linewidth=1)
axes[1].set_xlabel('Lag (months)  [positive = marketing leads]')
axes[1].set_ylabel('Cross-correlation')
axes[1].set_title('Cross-Correlation Function: Marketing Spend -> Sales')
axes[1].legend(fontsize=9)
axes[1].set_xlim(-20, 20)

plt.suptitle('Cross-Correlation: Detecting Leading/Lagging Indicators',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nPeak cross-correlation at lag: {peak_lag} months')
print(f'True lag in data generation:   {lag_true} months')
print('Cross-correlation successfully recovered the true leading lag.')


### Interpretation

The cross-correlation function peaks at **lag = 2**, which matches the true lag we built into the data. This tells us:

> *Marketing spend at time t is most correlated with sales at time t+2 — spend today predicts revenue two months from now.*

**Real-world applications:**
- **Finance:** Does GDP growth lead or lag unemployment?
- **Marketing:** What is the delay between ad spend and purchase conversion?
- **Healthcare:** How many days after an intervention does a biomarker respond?
- **Operations:** How far in advance does inventory need to be ordered relative to demand spikes?

> **Caution:** Cross-correlation at a particular lag does not prove causation. Two series may appear correlated across lags due to a shared trend or seasonal pattern. Always detrend and deseasonalise data before interpreting cross-correlations.


---

## 10. Regularised Regression on High-Dimensional Data

When we have many predictors, standard OLS can overfit — it finds spurious patterns in the training data that do not generalise. **Regularised regression** adds a penalty term to the loss function that shrinks coefficients:

| Method | Penalty | Effect |
|---|---|---|
| Ridge (L2) | alpha * sum(b_j^2) | Shrinks all coefficients towards zero; none exactly zero |
| Lasso (L1) | alpha * sum(|b_j|) | Shrinks some coefficients to **exactly zero** — feature selection |

The hyperparameter alpha controls regularisation strength. We will visualise how coefficients change as alpha increases — the **regularisation path**.


In [ ]:
# Diabetes dataset: OLS baseline vs Ridge vs Lasso
X_diab      = diabetes.drop(columns=['target']).values
y_diab      = diabetes['target'].values
feat_names  = diabetes.drop(columns=['target']).columns.tolist()

# Standardise features
scaler   = StandardScaler()
X_diab_s = scaler.fit_transform(X_diab)

X_tr, X_te, y_tr, y_te = train_test_split(X_diab_s, y_diab, test_size=0.2, random_state=42)

# OLS baseline
ols_diab    = LinearRegression().fit(X_tr, y_tr)
r2_ols_diab = r2_score(y_te, ols_diab.predict(X_te))
print(f'OLS baseline test R2: {r2_ols_diab:.4f}')
print()

# Ridge and Lasso at a few alpha values
for name, Model in [('Ridge', Ridge), ('Lasso', Lasso)]:
    print(f'{name}:')
    for alpha in [0.1, 1.0, 10.0]:
        m = Model(alpha=alpha, max_iter=10000).fit(X_tr, y_tr)
        r2  = r2_score(y_te, m.predict(X_te))
        nz  = np.sum(np.abs(m.coef_) > 1e-4)
        print(f'  alpha={alpha:5.1f}  test R2={r2:.4f}  non-zero coefs={nz}/{len(feat_names)}')
    print()


In [ ]:
# Compute regularisation paths
alphas = np.logspace(-2, 2, 100)

ridge_coefs = []
lasso_coefs = []

for a in alphas:
    ridge_coefs.append(Ridge(alpha=a).fit(X_tr, y_tr).coef_)
    lasso_coefs.append(Lasso(alpha=a, max_iter=10000).fit(X_tr, y_tr).coef_)

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_feat = plt.cm.tab10(np.linspace(0, 1, len(feat_names)))

for i, (feat, col) in enumerate(zip(feat_names, colors_feat)):
    axes[0].plot(alphas, ridge_coefs[:, i], color=col, linewidth=1.5, label=feat)
    axes[1].plot(alphas, lasso_coefs[:, i], color=col, linewidth=1.5, label=feat)

for ax, title in zip(axes,
    ['Ridge (L2) — All coefficients shrink, none reach zero',
     'Lasso (L1) — Coefficients hit zero = feature selection']):
    ax.set_xscale('log')
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Regularisation strength alpha (log scale)')
    ax.set_ylabel('Coefficient value')
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=7, loc='upper right', ncol=2)

plt.suptitle('Regularisation Paths: Ridge vs Lasso on Diabetes Dataset',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# Lasso feature selection at alpha=1.0
lasso_final = Lasso(alpha=1.0, max_iter=10000).fit(X_tr, y_tr)

coef_df = pd.DataFrame({
    'Feature':     feat_names,
    'OLS_coef':    ols_diab.coef_,
    'Lasso_coef':  lasso_final.coef_
}).set_index('Feature')
coef_df['Lasso_selected'] = (coef_df['Lasso_coef'].abs() > 1e-4)

print('Lasso (alpha=1.0) — feature selection result:')
print(coef_df.to_string())
print()
survivors = coef_df[coef_df['Lasso_selected']].index.tolist()
print(f'Features surviving Lasso at alpha=1.0: {survivors}')
print(f'That is {len(survivors)} out of {len(feat_names)} features retained.')


### Interpretation

The **regularisation path plots** tell a clear story:

- **Ridge (left):** As alpha increases, all coefficients shrink towards zero but **never reach it**. Ridge is a smoothing technique — it distributes weight across correlated features rather than selecting a subset.

- **Lasso (right):** As alpha increases, coefficients are driven to **exactly zero** one by one. The features whose lines hit zero first are the least important. The features still non-zero at high alpha are the most robustly predictive.

**Which to use?**

| Situation | Prefer |
|---|---|
| Many correlated features (e.g., genetic data) | Ridge — keeps all, shares weight |
| Want an interpretable sparse model | Lasso — automatic feature selection |
| Uncertain which features matter | Try both; use cross-validation for alpha |
| Best of both worlds | Elastic Net (L1 + L2 combined) |

> **Key insight:** Regularisation is not just a prediction trick — it is a statement about the *relationship* between features. Lasso says "only a subset of these variables truly relates to the outcome." Ridge says "all features contribute, but modestly."


---

## 11. Model Comparison Dashboard

Let's bring everything together — a summary table of all models built in this notebook, followed by a visual comparison of predicted vs actual values.


In [ ]:
# Summary table of all models
r2_ridge_1 = r2_score(y_te, Ridge(alpha=1.0).fit(X_tr, y_tr).predict(X_te))
r2_lasso_1 = r2_score(y_te, lasso_final.predict(X_te))

summary_data = [
    {'Model': 'Partial Correlation',       'Dataset': 'tips',     'Features': 'size (controlling total_bill)', 'Metric': f'r={r_partial:.3f}',              'Notes': 'Removes confounding by bill'},
    {'Model': 'MLR without interaction',   'Dataset': 'tips',     'Features': 'total_bill, time',              'Metric': f'R2={r2_no:.4f}',                 'Notes': 'Parallel slopes assumed'},
    {'Model': 'MLR with interaction',      'Dataset': 'tips',     'Features': 'total_bill x time',             'Metric': f'R2={r2_int:.4f}',                'Notes': 'Different slopes per meal time'},
    {'Model': 'Polynomial deg=1',          'Dataset': 'mpg',      'Features': 'horsepower',                    'Metric': f'test R2={poly_results[1]["r2_test"]:.4f}', 'Notes': 'Baseline linear'},
    {'Model': 'Polynomial deg=2',          'Dataset': 'mpg',      'Features': 'hp, hp^2',                      'Metric': f'test R2={poly_results[2]["r2_test"]:.4f}', 'Notes': 'Captures curvature'},
    {'Model': 'Polynomial deg=3',          'Dataset': 'mpg',      'Features': 'hp, hp^2, hp^3',                'Metric': f'test R2={poly_results[3]["r2_test"]:.4f}', 'Notes': 'Overfitting risk'},
    {'Model': 'Log-transformed linear',   'Dataset': 'mpg',      'Features': 'log(weight)',                   'Metric': f'R2={r2_log_w:.4f}',              'Notes': 'Better than raw weight'},
    {'Model': 'OLS (with outliers)',       'Dataset': 'tips*',    'Features': 'total_bill',                    'Metric': f'slope={ols.coef_[0]:.3f}',       'Notes': 'Pulled by 10 outliers'},
    {'Model': 'Huber regression',         'Dataset': 'tips*',    'Features': 'total_bill',                    'Metric': f'slope={huber.coef_[0]:.3f}',     'Notes': 'Robust to moderate outliers'},
    {'Model': 'RANSAC regression',        'Dataset': 'tips*',    'Features': 'total_bill',                    'Metric': f'slope={ransac.estimator_.coef_[0]:.3f}', 'Notes': 'Best outlier resistance'},
    {'Model': 'Logistic regression',      'Dataset': 'tips',     'Features': 'total_bill',                    'Metric': f'AUC={auc:.4f}',                  'Notes': f'OR={odds_ratio:.3f} per $1'},
    {'Model': 'Ridge (alpha=1.0)',        'Dataset': 'diabetes', 'Features': 'all 10',                        'Metric': f'test R2={r2_ridge_1:.4f}',       'Notes': 'All features retained'},
    {'Model': 'Lasso (alpha=1.0)',        'Dataset': 'diabetes', 'Features': f'{len(survivors)} selected',    'Metric': f'test R2={r2_lasso_1:.4f}',       'Notes': 'Sparse feature selection'},
]

summary_df = pd.DataFrame(summary_data)
pd.set_option('display.max_colwidth', 40)
print(summary_df.to_string(index=False))


In [ ]:
# 2x3 predicted vs actual mini-plots
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

panels = [
    ('MLR: no interaction (tips)',
     tips_int['tip'].values,
     lr_no.predict(X_no),
     '#e07b39',
     f'R2={r2_no:.3f}'),
    ('MLR: with interaction (tips)',
     tips_int['tip'].values,
     lr_int.predict(X_int),
     '#3a7ebf',
     f'R2={r2_int:.3f}'),
    ('Polynomial deg=2 (mpg)',
     y_hp,
     poly_results[2]['model'].predict(X_hp),
     '#2ca02c',
     f'R2={poly_results[2]["r2_train"]:.3f}'),
    ('Log-transform linear (mpg)',
     y_w,
     lr_log.predict(X_lw),
     '#9467bd',
     f'R2={r2_log_w:.3f}'),
    ('Huber robust (tips+outliers)',
     y_d,
     huber.predict(X_d),
     '#8c564b',
     f'slope={huber.coef_[0]:.3f}'),
    ('Ridge regression (diabetes)',
     y_te,
     Ridge(alpha=1.0).fit(X_tr, y_tr).predict(X_te),
     '#17becf',
     f'R2={r2_ridge_1:.3f}'),
]

for ax, (title, y_true, y_pred, color, metric) in zip(axes, panels):
    ax.scatter(y_true, y_pred, alpha=0.4, s=20, color=color,
               edgecolors='k', linewidths=0.2)
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=1, alpha=0.7)
    ax.set_xlabel('Actual')
    ax.set_ylabel('Predicted')
    ax.set_title(f'{title}\n{metric}', fontsize=9)

plt.suptitle('Model Comparison Dashboard: Predicted vs Actual',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---

## 12. Practice Exercises

Apply what you have learned. Try each exercise independently before revealing the solution.

---

### Exercise 1: Quadratic Party Size Effect

**Task:** Add a quadratic term for `size` (`size^2`) to the base MLR model that predicts `tip` from `total_bill` and `size`. Does the quadratic term improve R2? What does this mean?

**Hint:** Create a new column `size_sq = size ** 2` and add it to your feature matrix.


In [ ]:
# Your code here



In [ ]:
# Solution (Exercise 1) — uncomment to reveal

# tips_ex1 = tips.copy()
# tips_ex1['size_sq'] = tips_ex1['size'] ** 2
#
# # Baseline: tip ~ total_bill + size
# X_base_ex1 = tips_ex1[['total_bill', 'size']].values
# y_ex1      = tips_ex1['tip'].values
# lr_base_ex1  = LinearRegression().fit(X_base_ex1, y_ex1)
# r2_base_ex1  = r2_score(y_ex1, lr_base_ex1.predict(X_base_ex1))
#
# # Extended: tip ~ total_bill + size + size^2
# X_quad_ex1   = tips_ex1[['total_bill', 'size', 'size_sq']].values
# lr_quad_ex1  = LinearRegression().fit(X_quad_ex1, y_ex1)
# r2_quad_ex1  = r2_score(y_ex1, lr_quad_ex1.predict(X_quad_ex1))
#
# print(f'Baseline MLR (bill + size)         : R2 = {r2_base_ex1:.4f}')
# print(f'Extended MLR (bill + size + size^2) : R2 = {r2_quad_ex1:.4f}')
# print(f'R2 improvement: {r2_quad_ex1 - r2_base_ex1:+.4f}')
# print()
# print(f'size^2 coefficient: {lr_quad_ex1.coef_[2]:.4f}')
# print('A very small coefficient for size^2 with minimal R2 improvement')
# print('suggests the tip ~ size relationship is approximately linear here.')


---

### Exercise 2: Autocorrelation of Residuals

**Task:** Fit a linear regression of `mpg ~ horsepower + weight` on the mpg dataset. Then compute the autocorrelation of the residuals at lags 1–10. If the residuals are autocorrelated, what does that imply about the model?

**Hint:** Use `pd.Series(residuals).autocorr(lag=k)` in a loop, and optionally `plot_acf(residuals, lags=15)` for a visual check.


In [ ]:
# Your code here



In [ ]:
# Solution (Exercise 2) — uncomment to reveal

# mpg_ex2  = mpg.dropna(subset=['horsepower', 'weight', 'mpg']).copy()
# X_ex2    = mpg_ex2[['horsepower', 'weight']].values
# y_ex2    = mpg_ex2['mpg'].values
#
# lr_ex2   = LinearRegression().fit(X_ex2, y_ex2)
# resid_ex2 = pd.Series(y_ex2 - lr_ex2.predict(X_ex2))
#
# print('Autocorrelation of residuals:')
# for lag in range(1, 11):
#     ac  = resid_ex2.autocorr(lag=lag)
#     bar = '#' * int(abs(ac) * 30)
#     print(f'  lag {lag:2d}: r = {ac:+.3f}  {bar}')
#
# fig, ax = plt.subplots(figsize=(8, 3))
# plot_acf(resid_ex2.values, lags=20, ax=ax)
# ax.set_title('ACF of MLR Residuals (mpg ~ hp + weight)')
# plt.tight_layout()
# plt.show()
#
# print()
# print('If significant autocorrelation in residuals is found:')
# print('  -> The OLS independence assumption is VIOLATED')
# print('  -> Standard errors and p-values are unreliable')
# print('  -> Consider missing predictors, non-linear terms, or time-series models')


---

### Exercise 3: Logistic Regression with Controls

**Task:** Build a logistic regression that predicts `generous` (tip rate > 20%) using:
1. **Only** `day` (as dummies) — which day of the week has the highest odds of generous tipping?
2. **With** `total_bill` added as a control — do the day effects change after controlling for bill size?

**Hint:** Use `pd.get_dummies(tips['day'], drop_first=True)` to encode the day variable. Print or plot the coefficients (and odds ratios) from both models.


In [ ]:
# Your code here



In [ ]:
# Solution (Exercise 3) — uncomment to reveal

# tips_ex3  = tips_lr.copy()  # tips_lr already has 'generous' column
# day_dummies = pd.get_dummies(tips_ex3['day'], drop_first=True).astype(int)
# y_ex3     = tips_ex3['generous'].values
# day_feat_names = list(day_dummies.columns)
#
# # Model A: day only
# X_day_only  = day_dummies.values
# lr_day_only = LogisticRegression(random_state=42, max_iter=1000).fit(X_day_only, y_ex3)
# auc_day     = roc_auc_score(y_ex3, lr_day_only.predict_proba(X_day_only)[:,1])
#
# # Model B: day + total_bill
# X_day_bill  = np.column_stack([day_dummies.values, tips_ex3['total_bill'].values])
# lr_day_bill = LogisticRegression(random_state=42, max_iter=1000).fit(X_day_bill, y_ex3)
# auc_ctrl    = roc_auc_score(y_ex3, lr_day_bill.predict_proba(X_day_bill)[:,1])
#
# print('Model A (day only):')
# for f, c in zip(day_feat_names, lr_day_only.coef_[0]):
#     print(f'  {f}: coef={c:.3f}  OR={np.exp(c):.3f}')
# print(f'  AUC = {auc_day:.4f}')
# print()
# print('Model B (day + total_bill):')
# for f, c in zip(day_feat_names + ['total_bill'], lr_day_bill.coef_[0]):
#     print(f'  {f}: coef={c:.3f}  OR={np.exp(c):.3f}')
# print(f'  AUC = {auc_ctrl:.4f}')
# print()
# print('If day coefficients change substantially after controlling for total_bill,')
# print('part of the apparent day effect was actually a bill-size effect in disguise.')


---

## Summary

Congratulations on completing the enhanced tutorial. Here is what you covered beyond the base tutorial:

| Technique | Key Insight |
|---|---|
| **Partial correlation** | Removes confounding; use to isolate a direct relationship |
| **Interaction terms** | Tests whether the effect of X on Y changes with Z |
| **Polynomial regression** | Captures curves; compare train vs test R2 to avoid overfitting |
| **Log transformation** | Linearises exponential/skewed relationships |
| **Robust regression** | OLS is sensitive to outliers; Huber and RANSAC are resistant |
| **Logistic regression** | Extends regression to binary outcomes; interpret via odds ratios |
| **Autocorrelation** | Measures a series' relationship with itself over time; reveals seasonality |
| **Cross-correlation** | Detects leading/lagging relationships between two time series |
| **Ridge / Lasso** | Regularisation prevents overfitting; Lasso selects features |

**Next steps:**
- Try Elastic Net (combines Ridge + Lasso penalties)
- Explore time-series decomposition (`statsmodels.tsa.seasonal_decompose`)
- Learn about Generalised Linear Models (GLMs) — the unified framework behind linear and logistic regression
- Investigate causal inference methods (instrumental variables, difference-in-differences) for moving beyond correlation
